# Homework #2 — F1 Data Analysis with PySpark
Yixuan (Helle) Huang


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Q1. Average, slowest, and fastest pit stop time per driver per race

### Logic
The `pit_stops` table has one row per pit stop, with `raceId`, `driverId`, and `milliseconds` (the most precise duration field). A driver can pit multiple times in one race, so I group by `(raceId, driverId)` and compute three aggregates on `milliseconds`: mean (avg), max (slowest), and min (fastest). I then join `races` and `drivers` so the output is human-readable (race name + year + driver name) instead of raw IDs. I keep `milliseconds` as the unit because it is the rawest, most precise column; I also add a seconds version for readability.

In [0]:
# Q1: average / slowest / fastest pit stop per driver per race
pit_summary = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.round(F.avg("milliseconds"), 0).alias("avg_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms"),
        F.min("milliseconds").alias("fastest_pit_ms"),
        F.count("*").alias("n_stops"),
    )
    .withColumn("avg_pit_sec", F.round(F.col("avg_pit_ms") / 1000, 3))
)

q1 = (
    pit_summary
    .join(races.select("raceId", "year", F.col("name").alias("race_name")), "raceId")
    .join(drivers.select("driverId", "code", "forename", "surname"), "driverId")
    .select("year", "race_name", "driverId", "code", "forename", "surname",
            "n_stops", "avg_pit_ms", "avg_pit_sec", "fastest_pit_ms", "slowest_pit_ms")
    .orderBy("year", "race_name", "avg_pit_ms")
)

display(q1)

### Code explanation
- `groupBy("raceId", "driverId")` collapses all pit stops a driver made in a given race into one row.
- `F.avg`, `F.max`, `F.min` on `milliseconds` give the average, slowest, and fastest stop. I use `milliseconds` because it is the most precise raw field; the `duration` string column is less reliable.
- `F.count("*")` records how many stops the driver made — useful context for the average.
- The two `join` calls attach race metadata and driver names so the result is readable.
- `orderBy` sorts within each race from fastest to slowest average.

### Extra credit — alternative approach
An alternative is to use a **window function** instead of `groupBy`, which keeps every original pit-stop row but adds the aggregates alongside:
```python
w = Window.partitionBy("raceId", "driverId")
pit_stops.select(
    "raceId", "driverId", "stop", "milliseconds",
    F.avg("milliseconds").over(w).alias("avg_pit_ms"),
    F.min("milliseconds").over(w).alias("fastest_pit_ms"),
    F.max("milliseconds").over(w).alias("slowest_pit_ms"),
).show()
```
The window version is more memory-intensive but useful when you want stop-level detail and aggregates in the same dataframe (e.g., to flag which specific stop was the fastest).

## Q2. Rank average pit-stop time by finishing position

### Logic
I take the per-driver-per-race average from Q1 and join it to `results` to get each driver's `positionOrder` (finishing position) in that race. I then rank the average pit times **within each race** from fastest to slowest using a window.

**Handling DNFs:** in `results`, drivers who did not finish still have a `positionOrder` (it counts every classified entry), but their `position` field is null. I keep DNF drivers in the output but flag them with a `dnf` boolean (`position IS NULL`) and rank them at the bottom of their race using `positionOrder` as the tiebreaker. This way the ranking is transparent: a viewer can see both the pit-stop rank and whether the driver actually finished.

In [0]:
# Q2: rank average pit time within each race, ordered by finishing position
results_slim = results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("positionOrder"),
    F.col("position").alias("finish_position"),  # null = DNF
)

q2_base = (
    pit_summary  # from Q1
    .join(results_slim, ["raceId", "driverId"])
    .join(races.select("raceId", "year", F.col("name").alias("race_name")), "raceId")
    .join(drivers.select("driverId", "code", "surname"), "driverId")
    .withColumn("dnf", F.col("finish_position").isNull())
)

# rank fastest -> slowest avg pit time within each race
w_race = Window.partitionBy("raceId").orderBy(F.col("avg_pit_ms").asc())
q2 = (
    q2_base
    .withColumn("pit_rank", F.rank().over(w_race))
    .select("year", "race_name", "positionOrder", "finish_position", "dnf",
            "code", "surname", "avg_pit_ms", "pit_rank")
    .orderBy("year", "race_name", "positionOrder")
)

display(q2)

### Code explanation
- `results_slim` keeps only the columns I need and casts `positionOrder` to int for clean sorting.
- The join chain attaches finishing position, race info, and driver code to the Q1 averages.
- `withColumn("dnf", ...)` marks any row whose `position` is null — those are drivers who started but didn't finish.
- `Window.partitionBy("raceId").orderBy(avg_pit_ms.asc())` defines a per-race window sorted by average pit time; `F.rank()` then assigns 1 to the fastest average within each race, 2 to the next, etc.
- Final `orderBy("year", "race_name", "positionOrder")` displays the table in finishing order so you can visually compare finish position vs pit-stop rank.

### Comment on DNFs
DNFs are kept (not dropped) so the analysis is honest about who pitted but didn't finish. They are flagged via the `dnf` column and naturally fall to the bottom of `positionOrder`. An alternative would be to drop them with `.filter(F.col("finish_position").isNotNull())` if the question were strictly about classified finishers — I chose inclusion because pit performance is independent of whether the car later broke down.

### Extra credit — alternative approach
Instead of `rank()`, use `dense_rank()` (no gaps after ties) or `row_number()` (forces a strict ordering even on ties). For F1 pit data, ties on millisecond averages are rare, so all three give nearly identical output, but `dense_rank` is preferable when reporting "top-N fastest pitters" because tied drivers don't push others out of the top N.

## Q3. Fill missing 3-letter codes in the drivers dataset

### Logic
In the `drivers` table, the `code` column is null (or `\N`) for many older drivers. The F1 convention for the 3-letter code is the **first three letters of the surname, uppercased** (e.g., Alonso → ALO, Hamilton → HAM, Verstappen → VER). I will:
1. Detect missing codes — both true `null` and the literal string `\N` that CSVs sometimes contain.
2. Generate a candidate code from the surname.
3. Strip non-letters first (Räikkönen → RAI, da Matta → DAM... actually we want surname only after removing spaces/accents).
4. Coalesce: keep the existing code if present, otherwise use the generated one.

I document this rule so it's reproducible.

In [0]:
# Q3: fill missing driver codes
# Treat both nulls and the string "\N" as missing
drivers_clean = drivers.withColumn(
    "code_clean",
    F.when((F.col("code").isNull()) | (F.col("code") == r"\N"), None).otherwise(F.col("code"))
)

# Generate a 3-letter code from surname:
#   1. uppercase
#   2. keep only A-Z (drops spaces, hyphens, accents handled via regex_replace)
#   3. take first 3 chars
generated_code = F.upper(F.regexp_replace(F.col("surname"), "[^A-Za-z]", "")).substr(1, 3)

drivers_filled = drivers_clean.withColumn(
    "code_filled",
    F.coalesce(F.col("code_clean"), generated_code)
)

# Show the rows where we actually filled something in
q3 = drivers_filled.filter(F.col("code_clean").isNull()).select(
    "driverId", "forename", "surname", "code", "code_filled"
)

print(f"Drivers missing a code originally: {q3.count()}")
display(q3)

### Code explanation
- The first `withColumn` normalizes "missing": both real nulls and the literal `\N` string (a common CSV artifact) become `None`.
- `regexp_replace(surname, "[^A-Za-z]", "")` strips anything that isn't a letter, so "de la Rosa" becomes "delaRosa" before truncation. I then `upper()` and `substr(1, 3)` to take the first 3 letters.
- `F.coalesce(code_clean, generated_code)` keeps the official code when it exists and falls back to the generated one otherwise — this preserves real F1 codes (which sometimes deviate from the surname rule, e.g., Magnussen = MAG, Verstappen = VER) and only invents codes where the dataset is silent.
- The final filter shows only the rows where we actually invented a code, so the grader can audit them.

### Caveat
This rule will occasionally collide (e.g., two drivers with surnames starting "MAR"). In a production fix I would detect collisions with a `groupBy("code_filled").count()` and disambiguate manually. For this assignment the surname-prefix rule matches F1's own convention closely enough.

### Extra credit — alternative approach
A more sophisticated alternative is to **check for collisions and use the 4th letter as a tiebreaker** for the second occurrence:
```python
w = Window.partitionBy("code_filled").orderBy("driverId")
drivers_filled.withColumn("dup_idx", F.row_number().over(w)) \
    .withColumn("code_final",
        F.when(F.col("dup_idx") == 1, F.col("code_filled"))
         .otherwise(F.upper(F.substring("surname", 1, 2)) + F.upper(F.substring("forename", 1, 1)))
    )
```
This mimics how F1 historically resolved code clashes (e.g., using initials).

## Q4. Youngest and oldest driver per race

### Definition of Age
I define **age** as *the number of birthdays the driver had completed before the race date* — i.e., full years lived. Concretely: `age = year(race_date) - year(dob)`, then subtract 1 if the race date falls **before** the driver's birthday in the race year (because they haven't had that year's birthday yet). This matches the everyday meaning of "how old are you today."

In [0]:
# Q4: age per driver per race, then youngest/oldest per race
race_dates = races.select("raceId", F.col("date").cast("date").alias("race_date"),
                          "year", F.col("name").alias("race_name"))
driver_dob = drivers.select("driverId", "forename", "surname",
                            F.col("dob").cast("date").alias("dob"))

driver_race = (
    results.select("raceId", "driverId").distinct()
    .join(race_dates, "raceId")
    .join(driver_dob, "driverId")
)

# Age = completed birthdays as of race_date
driver_race_age = driver_race.withColumn(
    "Age",
    F.year("race_date") - F.year("dob") -
    F.when(
        (F.month("race_date") < F.month("dob")) |
        ((F.month("race_date") == F.month("dob")) & (F.dayofmonth("race_date") < F.dayofmonth("dob"))),
        1
    ).otherwise(0)
)

# Youngest and oldest per race using window functions
w_young = Window.partitionBy("raceId").orderBy(F.col("Age").asc())
w_old   = Window.partitionBy("raceId").orderBy(F.col("Age").desc())

q4 = (
    driver_race_age
    .withColumn("rk_young", F.row_number().over(w_young))
    .withColumn("rk_old",   F.row_number().over(w_old))
    .filter((F.col("rk_young") == 1) | (F.col("rk_old") == 1))
    .withColumn("label", F.when(F.col("rk_young") == 1, F.lit("youngest")).otherwise(F.lit("oldest")))
    .select("year", "race_name", "label", "forename", "surname", "dob", "race_date", "Age")
    .orderBy("year", "race_name", "label")
)

q4.display(20, truncate=False)

### Code explanation
- I cast `races.date` and `drivers.dob` to `date` so the month/day comparisons are valid.
- The `Age` formula starts with `year(race_date) - year(dob)` and subtracts 1 if the driver's birthday hasn't occurred yet in the race year. The `F.when` clause encodes the "before birthday" check by comparing month first and then day for same-month cases.
- I then build two windows partitioned by `raceId`: one ascending (youngest) and one descending (oldest). `row_number()` picks exactly one driver per race per window, breaking ties arbitrarily (by row order).
- The final filter keeps only rank-1 rows from either window and labels them.

### Extra credit — alternative approach
An alternative uses Spark's built-in `months_between` for an exact fractional age and then floors it:
```python
F.floor(F.months_between("race_date", "dob") / 12).alias("Age")
```
This avoids the manual month/day arithmetic and is one of the cleanest one-liners. I chose the explicit version above because it makes the "completed birthdays" definition obvious in the code itself.

## Q5. Cumulative wins / 2nd / 3rd places per driver going into each race

### Logic
For each (driver, race), I want three counts that reflect the driver's career **before** the current race: how many 1st-, 2nd-, and 3rd-place finishes they have accumulated up to but **not including** that race. I do this with a window partitioned by `driverId` and ordered chronologically by race date, using `rowsBetween(unboundedPreceding, -1)` so the current race itself is excluded. Inside the window I sum three indicator columns (`is_1st`, `is_2nd`, `is_3rd`).

In [0]:
# Q5: cumulative wins, 2nds, 3rds per driver going into each race
res_with_date = (
    results.select("raceId", "driverId", F.col("positionOrder").cast("int").alias("pos"))
    .join(races.select("raceId", F.col("date").cast("date").alias("race_date"),
                       "year", F.col("name").alias("race_name")), "raceId")
    .withColumn("is_1st", (F.col("pos") == 1).cast("int"))
    .withColumn("is_2nd", (F.col("pos") == 2).cast("int"))
    .withColumn("is_3rd", (F.col("pos") == 3).cast("int"))
)

w_career = (
    Window.partitionBy("driverId")
          .orderBy("race_date")
          .rowsBetween(Window.unboundedPreceding, -1)  # strictly before current race
)

q5 = (
    res_with_date
    .withColumn("wins_before",  F.coalesce(F.sum("is_1st").over(w_career), F.lit(0)))
    .withColumn("p2_before",    F.coalesce(F.sum("is_2nd").over(w_career), F.lit(0)))
    .withColumn("p3_before",    F.coalesce(F.sum("is_3rd").over(w_career), F.lit(0)))
    .join(drivers.select("driverId", "forename", "surname"), "driverId")
    .select("year", "race_name", "race_date", "forename", "surname",
            "pos", "wins_before", "p2_before", "p3_before")
    .orderBy("race_date", "surname")
)

q5.display(20, truncate=False)

### Code explanation
- I create three integer indicator columns (`is_1st`, `is_2nd`, `is_3rd`) that are 1 when the driver finished in that position and 0 otherwise — casting a boolean comparison to int is the cleanest idiom.
- The window partitions on `driverId` (so each driver has their own running total) and orders chronologically by `race_date`. The crucial part is `rowsBetween(unboundedPreceding, -1)`: this sums everything **before** the current row, so the count reflects the driver's record going **into** the race, not after.
- `coalesce(..., 0)` replaces the null produced for a driver's very first race (no rows before it) with 0.

### Interpretation note
If the question instead means *cumulative through and including the current race*, just change the window to `rowsBetween(unboundedPreceding, 0)` or drop the `rowsBetween` clause entirely (Spark's default for ordered windows is `unboundedPreceding` to `currentRow`).

### Extra credit — alternative approach
A non-window alternative is a **self-join**: for each `(driverId, race_date)` row, join to all earlier results for that driver and group-count. This is conceptually clearer but ~O(n²) in the worst case and far slower than the window version on large data.

## Q6. My own question — Which circuits produce the most position changes from grid to finish?

### Question
Some F1 circuits are notorious for being processional (Monaco), while others are famous for overtaking (Interlagos, Spa). I want to **rank circuits by the average absolute change in position from grid to finish** — a simple proxy for how much "racing" actually happens.

### Logic
For each result, compute `abs(grid - positionOrder)` (excluding rows where grid is 0, which means a pit-lane start, and DNFs where the comparison is misleading). Average this per `circuitId`, join in circuit names, and sort descending. Higher average = more position shuffling.

In [0]:
# Q6: average absolute grid->finish position change per circuit
overtake = (
    results.select("raceId", "driverId",
                   F.col("grid").cast("int").alias("grid"),
                   F.col("positionOrder").cast("int").alias("finish"),
                   F.col("position").alias("classified"))
    .filter((F.col("grid") > 0) & F.col("classified").isNotNull())
    .withColumn("pos_change", F.abs(F.col("grid") - F.col("finish")))
    .join(races.select("raceId", "circuitId", "year"), "raceId")
    .join(circuits.select("circuitId", F.col("name").alias("circuit_name"), "country"), "circuitId")
)

q6 = (
    overtake
    .groupBy("circuitId", "circuit_name", "country")
    .agg(
        F.round(F.avg("pos_change"), 2).alias("avg_pos_change"),
        F.count("*").alias("n_classified_finishes"),
        F.countDistinct("raceId").alias("n_races"),
    )
    .filter(F.col("n_races") >= 5)  # only circuits with a meaningful sample
    .orderBy(F.col("avg_pos_change").desc())
)

q6.display(15, truncate=False)

### Code explanation
- I filter out grid `0` (pit-lane starts artificially inflate position change) and DNFs (`position` is null) so the metric reflects clean racing only.
- `pos_change = |grid - finish|` is the per-driver-per-race position swing.
- After joining race → circuit metadata, I average `pos_change` per circuit, count how many classified finishes contributed, and count how many distinct races were held there.
- The `n_races >= 5` filter prevents one-off circuits from topping the chart on noise.
- Sorting descending puts the most "overtake-friendly" circuits first.

### Interpretation
I expect circuits like Interlagos, Spa, and the old Nürburgring to rank high, and street circuits like Monaco and Singapore to rank low — which would validate the F1 fan-lore that street circuits are processional. The output of this cell answers that empirically.